In [2]:
import os
import sys

project_root = r"C:\Users\Olga\Диплом\Мой проект"
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

print(sys.path[-3:])

['C:\\Users\\Olga\\anaconda3\\Lib\\site-packages\\win32\\lib', 'C:\\Users\\Olga\\anaconda3\\Lib\\site-packages\\Pythonwin', 'C:\\Users\\Olga\\Диплом\\Мой проект\\src']


In [3]:
import pandas as pd
from sqlalchemy import text
from data_loader import get_engine

In [4]:
engine = get_engine()

In [5]:
query = """
    SELECT
        year,
        month,
        metric_code,
        value,
        monthly_value
    FROM crime_spb2
    WHERE year = 2025
      AND month <= 7
    ORDER BY metric_code, month
"""

crime_2025 = pd.read_sql(query, con=engine)
crime_2025

,year,month,metric_code,value,monthly_value
0,2025,1,drugs_total,None,820.285714
1,2025,2,drugs_total,None,820.285714
2,2025,3,drugs_total,None,820.285714
3,2025,4,drugs_total,None,820.285714
4,2025,5,drugs_total,None,820.285714
5,2025,6,drugs_total,None,820.285714
6,2025,7,drugs_total,None,820.285714
7,2025,1,econ_total,None,508.714286
8,2025,2,econ_total,None,508.714286
9,2025,3,econ_total,None,508.714286


In [6]:
crime_2025 = crime_2025.copy()

base_weights = [0.96, 0.98, 1.00, 1.01, 1.02, 1.00, 1.03]
weight_sum = sum(base_weights)
normalized_weights = [w * 7 / weight_sum for w in base_weights]

for metric_code in crime_2025["metric_code"].unique():
    mask = crime_2025["metric_code"] == metric_code
    metric_data = crime_2025.loc[mask].sort_values("month").copy()

    base = metric_data["monthly_value"].iloc[0]
    total_for_7_months = base * 7

    new_values = [total_for_7_months * w / 7 for w in normalized_weights]

    crime_2025.loc[metric_data.index, "monthly_value"] = new_values

crime_2025

,year,month,metric_code,value,monthly_value
0,2025,1,drugs_total,None,787.474286
1,2025,2,drugs_total,None,803.880000
2,2025,3,drugs_total,None,820.285714
3,2025,4,drugs_total,None,828.488571
4,2025,5,drugs_total,None,836.691429
5,2025,6,drugs_total,None,820.285714
6,2025,7,drugs_total,None,844.894286
7,2025,1,econ_total,None,488.365714
8,2025,2,econ_total,None,498.540000
9,2025,3,econ_total,None,508.714286


In [7]:
from sqlalchemy import text

with engine.begin() as connection:
    for _, row in crime_2025.iterrows():
        connection.execute(
            text("""
                UPDATE crime_spb2
                SET monthly_value = :monthly_value
                WHERE year = :year
                  AND month = :month
                  AND metric_code = :metric_code
            """),
            {
                "monthly_value": float(row["monthly_value"]),
                "year": int(row["year"]),
                "month": int(row["month"]),
                "metric_code": row["metric_code"],
            }
        )

In [8]:
check = pd.read_sql("""
    SELECT
        year,
        month,
        metric_code,
        value,
        monthly_value
    FROM crime_spb2
    WHERE year = 2025
      AND month <= 7
    ORDER BY metric_code, month
""", con=engine)

check

,year,month,metric_code,value,monthly_value
0,2025,1,drugs_total,None,787.474286
1,2025,2,drugs_total,None,803.880000
2,2025,3,drugs_total,None,820.285714
3,2025,4,drugs_total,None,828.488571
4,2025,5,drugs_total,None,836.691429
5,2025,6,drugs_total,None,820.285714
6,2025,7,drugs_total,None,844.894286
7,2025,1,econ_total,None,488.365714
8,2025,2,econ_total,None,498.540000
9,2025,3,econ_total,None,508.714286
